<a href="https://colab.research.google.com/github/HLZHarry/AlphaLink-Muti-Modal-RAG/blob/main/Step_2_The_File_Reorganzier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from pathlib import Path
import shutil
import os
from google.colab import drive

In [15]:
# 1. Force-delete the local 'drive' folder that's causing the error
!rm -rf /content/drive

# 2. Immediately try to mount the REAL Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 3. Check the path again
import os
check_path = "/content/drive/MyDrive/AlphaLink-RAG/data/raw"
if os.path.exists(check_path):
    print(f"✅ REAL DRIVE LINKED! I can see your data at: {check_path}")
    print("Contents:", os.listdir(check_path))
else:
    print("❌ Path still not found. Please look at your sidebar: What is the EXACT name of the folder inside 'MyDrive'?")

Mounted at /content/drive
✅ REAL DRIVE LINKED! I can see your data at: /content/drive/MyDrive/AlphaLink-RAG/data/raw
Contents: ['sec-edgar-filings']


In [16]:
# 1. Setup Paths (Matching your Step 1)
BASE_DIR = Path("/content/drive/MyDrive/AlphaLink-RAG")
# This is where the downloader SAVED the files
SEARCH_ROOT = BASE_DIR / "data/raw"
# This is where we WANT the clean files to go
DEST_DIR = BASE_DIR / "data/processed_filings"

DEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"🔍 Searching for filings in: {SEARCH_ROOT}...")

🔍 Searching for filings in: /content/drive/MyDrive/AlphaLink-RAG/data/raw...


In [17]:
# 2. Hunt for the downloader's output folder
# The library always creates a folder called 'sec-edgar-filings'
found_files = 0

# We use rglob to find the actual documents deeply nested in Ticker/Form/Accession folders
# Common files are 'full-submission.txt' or '.htm' files
for file_path in SEARCH_ROOT.rglob("*"):
    # We only want actual filing files (usually > 5KB to skip metadata)
    if file_path.is_file() and file_path.stat().st_size > 5000:
        # Avoid moving files that are already in the destination
        if "processed_filings" in str(file_path):
            continue

        try:
            # Create a clean name: TICKER_FORM_ACCESSION.txt
            # Parts usually look like: [..., 'sec-edgar-filings', 'NVDA', '10-K', '000...', 'full-submission.txt']
            parts = file_path.parts
            ticker = parts[-4]
            form = parts[-3]
            accession = parts[-2]

            new_name = f"{ticker}_{form}_{accession}{file_path.suffix}"
            shutil.copy2(file_path, DEST_DIR / new_name)
            found_files += 1
        except Exception as e:
            # If the path structure is different, we skip or log
            continue

if found_files > 0:
    print(f"✅ SUCCESS! Moved {found_files} files to {DEST_DIR}")
    print("Files ready for Step 3 (Parsing).")
else:
    print("❌ Still nothing found.")
    print("Let's see what IS in your raw folder. Running 'ls'...")
    !ls -R {SEARCH_ROOT} | head -n 20

✅ SUCCESS! Moved 442 files to /content/drive/MyDrive/AlphaLink-RAG/data/processed_filings
Files ready for Step 3 (Parsing).


- **`parents=True`** — Tells Python: *"If the middle folders are missing (like `/data/`), create those too."*

- **`exist_ok=True`** — The **"Safety Switch."** Prevents the script from crashing if the folder already exists.

- **`rglob("*")`** — A **Recursive Glob.** Tells Python to dive into every single subfolder inside `raw_dir` and find every file (`*`). Think of it as a high-speed search through a filing cabinet.

- **`is_file()`** — Ignores folders and only processes actual documents.

- **`st_size > 5000`** — Ignores tiny "junk" files (under 5KB) that are often just empty headers or metadata.

- **`.parts`** — The most clever piece. Splits the file path into a list of names. By counting backwards from the end (`-4`, `-3`), we can grab the **Ticker** and **Form type** regardless of how deep they were buried by the SEC downloader.

- **`shutil.copy2`** — The `2` version preserves the original file **metadata** (like the filing date). This is critical for our RAG — we want the AI to know exactly *when* NVIDIA said they were low on chips.